# LiDAR 場景辨識 CNN 訓練（專題資料）

用專題收集的雷射掃描影像（corner／corridor／right_corner／room，各 100 張），訓練一個 CNN 來辨識自走車目前所在的場景，並算出準確率與混淆矩陣。

**使用步驟**
1. 上方選單「執行階段 → 變更執行階段類型」選 **T4 GPU**（沒有也可以，只是比較慢）。
2. 在電腦上把 `程式碼/ros_ws/src/my_robot/dataset` 資料夾壓縮成 **dataset.zip**。
3. 從上到下依序按每一格左邊的 ▶ 執行；第 2 格會跳出上傳按鈕，選 dataset.zip。
4. 跑完後，最下面會顯示準確率、混淆矩陣，並下載新的模型檔。

In [ ]:
# 1. 載入套件
import os, glob, json, zipfile, random
import numpy as np
import cv2
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('使用裝置：', device)

In [ ]:
# 2. 上傳 dataset.zip 並解壓縮
DATA_DIR = None
for cand in ['dataset', 'data/dataset']:
    if os.path.isdir(cand):
        DATA_DIR = cand
if DATA_DIR is None:
    try:
        from google.colab import files
        up = files.upload()                      # 選 dataset.zip
        zname = [k for k in up if k.endswith('.zip')][0]
        with zipfile.ZipFile(zname) as z:
            z.extractall('data')
    except ImportError:
        raise SystemExit('請把 dataset 資料夾放在這個筆記本旁邊')
    # 找出「裡面直接有類別資料夾」的那一層
    for root, dirs, fs in os.walk('data'):
        if dirs and all(glob.glob(os.path.join(root, d, '*.png')) for d in dirs):
            DATA_DIR = root; break
print('資料夾：', DATA_DIR)

In [ ]:
# 3. 讀取影像（依檔名排序 = 收集時的時間順序）
CLASSES = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print('類別：', CLASSES)

images, labels, order = [], [], []
for ci, c in enumerate(CLASSES):
    fs = sorted(glob.glob(os.path.join(DATA_DIR, c, '*.png')))
    for k, f in enumerate(fs):
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (128, 128))
        images.append(img); labels.append(ci); order.append(k / max(len(fs) - 1, 1))
    print(f'{c:>13}: {len(fs)} 張')
images = np.stack(images).astype(np.float32) / 255.0
labels = np.array(labels); order = np.array(order)

fig, axes = plt.subplots(len(CLASSES), 5, figsize=(10, 2.2 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    idx = np.where(labels == r)[0]
    for j, i in enumerate(np.linspace(idx[0], idx[-1], 5).astype(int)):
        axes[r, j].imshow(images[i], cmap='gray'); axes[r, j].axis('off')
    axes[r, 0].set_title(c, loc='left')
plt.tight_layout(); plt.show()

### 為什麼不隨機切分資料？
每一類的 100 張是每 0.3 秒連拍的，相鄰兩張幾乎一樣。如果隨機切，測試資料會跟訓練資料「長得幾乎一樣」，準確率會虛高。

所以這裡依**時間順序**切：每一類**前 70% 當訓練、後 30% 當測試**，比較接近「在沒看過的位置辨識」的真實情況。

In [ ]:
# 4. 依時間順序切分訓練／測試
TRAIN_RATIO = 0.7
train_idx = np.where(order < TRAIN_RATIO)[0]
test_idx  = np.where(order >= TRAIN_RATIO)[0]
Xtr, ytr = torch.tensor(images[train_idx])[:, None], torch.tensor(labels[train_idx])
Xte, yte = torch.tensor(images[test_idx])[:, None],  torch.tensor(labels[test_idx])
print('訓練：', len(Xtr), '張；測試：', len(Xte), '張')

In [ ]:
# 5. 建立 CNN 模型（輕量，方便放到 Tinker Board 上跑）
class LidarCNN(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 64))  # 128 -> 8
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, n_cls))
    def forward(self, x):
        return self.head(self.features(x))

model = LidarCNN(len(CLASSES)).to(device)
print('參數量：', sum(p.numel() for p in model.parameters()))

def augment(x):
    # 資料擴增：隨機小角度旋轉（±10°）與平移（±4 px），模擬車子朝向與位置的小變化
    n = x.shape[0]
    ang = (torch.rand(n, device=x.device) - 0.5) * 2 * np.deg2rad(10)
    sh = (torch.rand(n, 2, device=x.device) - 0.5) * 2 * (4 / 64)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([cos, -sin, sh[:, 0]], 1), torch.stack([sin, cos, sh[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, align_corners=False)

In [ ]:
# 6. 訓練
EPOCHS, BATCH = 30, 32
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
hist = {'train_acc': [], 'test_acc': [], 'loss': []}

def evaluate(X, y):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(X), 128):
            preds.append(model(X[i:i+128].to(device)).argmax(1).cpu())
    p = torch.cat(preds); return (p == y).float().mean().item(), p

for ep in range(1, EPOCHS + 1):
    model.train(); perm = torch.randperm(len(Xtr)); tot = 0
    for i in range(0, len(Xtr), BATCH):
        b = perm[i:i+BATCH]
        xb, yb = augment(Xtr[b].to(device)), ytr[b].to(device)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
    tr, _ = evaluate(Xtr, ytr); te, _ = evaluate(Xte, yte)
    hist['loss'].append(tot / len(Xtr)); hist['train_acc'].append(tr); hist['test_acc'].append(te)
    if ep % 5 == 0 or ep == 1:
        print(f'第 {ep:2d} 輪  loss={tot/len(Xtr):.3f}  訓練準確率={tr:.1%}  測試準確率={te:.1%}')

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist['loss']); ax[0].set_title('Training loss'); ax[0].set_xlabel('epoch')
ax[1].plot(hist['train_acc'], label='train'); ax[1].plot(hist['test_acc'], label='test')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('training_curve.png', dpi=150); plt.show()

In [ ]:
# 7. 結果：準確率、混淆矩陣、各類別表現
acc, pred = evaluate(Xte, yte)
print(f'★ 測試資料準確率：{acc:.1%}（{int((pred == yte).sum())}/{len(yte)} 張）\n')
print(classification_report(yte, pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(yte, pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues', colorbar=False)
plt.title(f'Confusion matrix (test acc {acc:.1%})'); plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150); plt.show()

wrong = np.where(pred.numpy() != yte.numpy())[0]
print('辨識錯誤：', len(wrong), '張')
if len(wrong):
    k = min(8, len(wrong)); fig, axes = plt.subplots(1, k, figsize=(2 * k, 2.4)); axes = np.atleast_1d(axes)
    for a, i in zip(axes, wrong[:k]):
        a.imshow(Xte[i, 0], cmap='gray'); a.axis('off')
        a.set_title(f'true: {CLASSES[yte[i]]}\npred: {CLASSES[pred[i]]}', fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# 8. 匯出模型（格式與專題的 ai_observer.py 相容：輸入 1x1x128x128、數值 0~1）
!pip install -q onnx
model.eval().cpu()
torch.onnx.export(model, torch.zeros(1, 1, 128, 128), 'lidar_brain_v2.onnx',
                  input_names=['input'], output_names=['logits'], opset_version=13, dynamo=False)
with open('classes.json', 'w') as f:
    json.dump(CLASSES, f)
print('已產生：lidar_brain_v2.onnx、classes.json、training_curve.png、confusion_matrix.png')
try:
    from google.colab import files
    for fn in ['lidar_brain_v2.onnx', 'classes.json', 'training_curve.png', 'confusion_matrix.png']:
        files.download(fn)
except ImportError:
    pass

## 跑完之後
把這三個數字記下來，告訴 Claude，就可以寫進備審：
1. **測試準確率**（第 7 格的 ★ 那行）
2. **哪兩類最容易搞混**（看混淆矩陣）
3. `confusion_matrix.png` 和 `training_curve.png` 兩張圖（可以放進專題頁）

備註：每次訓練結果會有些微差異，這是正常的。